<a href="https://colab.research.google.com/github/avyue/datasci112_finalproject/blob/main/Prepping_Models_(parts_a%20through%20d_of_Prediction).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
#IMPORT PACKAGES AND SET SEED

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    log_loss, accuracy_score, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 226
N_FOLDS = 5
THRESHOLD = 0.5

In [14]:
#LOAD THE DATASET

df = pd.read_csv("explore_80.csv")

# Treat blank or whitespace-only cells as missing.
df = df.replace(r"^\s*$", np.nan, regex=True)

# Separate the outcome from the other columns.
y = df["Netflix"].astype(int)
X = df.drop(columns="Netflix").copy()

# Find the genre indicator columns.
genre_columns = [
    column for column in X.columns
    if column.startswith("genre_")
]

# Identify titles with unknown genre information.
X["genres_missing"] = (
    X["genres"].fillna("").str.strip().eq("").astype(int)
)

# The missingness indicator distinguishes these zeros
# from known genre absences.
X.loc[X["genres_missing"] == 1, genre_columns] = 0

# Keep missing age classifications as a separate category.
X["Age"] = X["Age"].fillna("unrated")

print(f"Development titles: {len(df):,}")
print(f"Titles on Netflix: {y.sum():,}")
print(f"Netflix percentage: {100 * y.mean():.2f}%")

missing_runtime = X["runtime_minutes"].isna().sum()
print(f"Missing runtimes: {missing_runtime:,}")
print(f"Percentage missing: {100 * missing_runtime / len(df):.2f}%")

Development titles: 11,906
Titles on Netflix: 4,491
Netflix percentage: 37.72%
Missing runtimes: 3,083
Percentage missing: 25.89%


In [15]:
#CREATE THE FIVE FOLDS ONCE

cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

cv_splits = list(cv.split(X, y))

for fold, (train_index, validation_index) in enumerate(
    cv_splits, start=1
):
    print(
        f"Fold {fold}: "
        f"{len(train_index):,} training titles, "
        f"{len(validation_index):,} validation titles"
    )

Fold 1: 9,524 training titles, 2,382 validation titles
Fold 2: 9,525 training titles, 2,381 validation titles
Fold 3: 9,525 training titles, 2,381 validation titles
Fold 4: 9,525 training titles, 2,381 validation titles
Fold 5: 9,525 training titles, 2,381 validation titles


In [16]:
#DEFINE PREPROCESSING

class RuntimeMedianImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        # Learn replacement values from training data only.
        runtimes = pd.to_numeric(
            X["runtime_minutes"], errors="coerce"
        )

        self.medians_ = runtimes.groupby(
            X["content_type"]
        ).median()

        # Fallback if a content type has no observed training runtimes.
        self.fallback_ = runtimes.median()

        if pd.isna(self.fallback_):
            raise ValueError("No observed training runtimes.")

        return self

    def transform(self, X):
        out = X.copy()

        runtimes = pd.to_numeric(
            out["runtime_minutes"], errors="coerce"
        )

        # Record missingness before filling values.
        out["runtime_missing"] = runtimes.isna().astype(int)

        replacements = (
            out["content_type"]
            .map(self.medians_)
            .fillna(self.fallback_)
        )

        out["runtime_minutes"] = runtimes.fillna(replacements)

        return out


def make_preprocessor(scale_numeric=True):

    # Standardize runtime for logistic regression.
    # Trees can use the original runtime scale.
    numeric_transform = (
        StandardScaler() if scale_numeric else "passthrough"
    )

    binary_columns = [
        "Type",
        "runtime_missing",
        "genres_missing"
    ] + genre_columns

    encoding = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_transform,
                ["runtime_minutes"]
            ),
            (
                "age",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["Age"]
            ),
            (
                "binary",
                "passthrough",
                binary_columns
            )
        ],
        # Exclude IDs, Netflix dates, other platform flags,
        # and any columns not explicitly selected above.
        remainder="drop"
    )

    return Pipeline([
        ("runtime", RuntimeMedianImputer()),
        ("encoding", encoding)
    ])

In [17]:
#RUN PREPROCESSING AND EVALUATE THE BASELINE

baseline_rows = []
imputation_rows = []

for fold, (train_index, validation_index) in enumerate(
    cv_splits, start=1
):
    X_train = X.iloc[train_index].copy()
    X_validation = X.iloc[validation_index].copy()

    y_train = y.iloc[train_index]
    y_validation = y.iloc[validation_index]

    # PART B: Fit preprocessing on training data only.
    preprocessor = make_preprocessor(scale_numeric=True)

    prepared_train = preprocessor.fit_transform(X_train)

    # Apply the existing preprocessing to validation data.
    prepared_validation = preprocessor.transform(X_validation)

    # Check that the prepared inputs contain no missing values.
    assert np.isfinite(prepared_train).all()
    assert np.isfinite(prepared_validation).all()

    # Record the runtime medians learned in this fold.
    imputer = preprocessor.named_steps["runtime"]

    imputation_rows.append({
        "Fold": fold,
        "Movie median": imputer.medians_["Movie"],
        "TV median": imputer.medians_["TV Show"],
        "Validation runtimes filled":
            X_validation["runtime_minutes"].isna().sum()
    })

    # PART C: Fit the baseline.
    # A column of zeros makes clear that no title features are used.
    baseline = DummyClassifier(strategy="prior")

    baseline.fit(
        np.zeros((len(train_index), 1)),
        y_train
    )

    positive_column = np.flatnonzero(
        baseline.classes_ == 1
    )[0]

    probabilities = baseline.predict_proba(
        np.zeros((len(validation_index), 1))
    )[:, positive_column]

    # Convert probabilities to classifications only for
    # misclassification rate, precision, and recall.
    predictions = (probabilities >= THRESHOLD).astype(int)

    # Precision is undefined if no positive predictions are made.
    precision = (
        precision_score(y_validation, predictions)
        if predictions.sum() > 0
        else np.nan
    )

    baseline_rows.append({
        "Fold": fold,
        "Training Netflix proportion": y_train.mean(),
        "Log loss": log_loss(
            y_validation, probabilities, labels=[0, 1]
        ),
        "Misclassification rate":
            1 - accuracy_score(y_validation, predictions),
        "Precision": precision,
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        )
    })

imputation_results = pd.DataFrame(imputation_rows)
baseline_results = pd.DataFrame(baseline_rows)

In [18]:
#DISPLAY RESULTS FOR PARTS B AND C OF REPORT

print("RUNTIME IMPUTATION BY FOLD")
print(imputation_results.to_string(index=False))

print("\nBASELINE RESULTS BY FOLD")
print(
    baseline_results.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nSUMMARY")

print(
    "Mean validation log loss:",
    f"{baseline_results['Log loss'].mean():.6f}"
)

print(
    "Standard deviation of log loss across folds:",
    f"{baseline_results['Log loss'].std(ddof=1):.6f}"
)

print(
    "Mean misclassification rate:",
    f"{100 * baseline_results['Misclassification rate'].mean():.2f}%"
)

print(
    "Mean recall:",
    f"{100 * baseline_results['Recall'].mean():.2f}%"
)

if baseline_results["Precision"].isna().all():
    print("Precision: undefined because no positive predictions were made")
else:
    print("Precision by fold:", baseline_results["Precision"].tolist())

RUNTIME IMPUTATION BY FOLD
 Fold  Movie median  TV median  Validation runtimes filled
    1          95.0       40.0                         610
    2          95.0       40.0                         587
    3          95.0       40.0                         612
    4          95.0       40.0                         631
    5          96.0       41.0                         643

BASELINE RESULTS BY FOLD
 Fold  Training Netflix proportion  Log loss  Misclassification rate  Precision   Recall
    1                     0.377152  0.662784                0.377414        NaN 0.000000
    2                     0.377218  0.662653                0.377152        NaN 0.000000
    3                     0.377218  0.662653                0.377152        NaN 0.000000
    4                     0.377218  0.662653                0.377152        NaN 0.000000
    5                     0.377218  0.662653                0.377152        NaN 0.000000

SUMMARY
Mean validation log loss: 0.662679
Standard deviat

In [19]:
#RECORD PROPOSED TUNING SETTINGS (records part b plan; does not fit 3 predictive models yet)

TUNING_PLAN = {
    "logistic_regression": {
        "regularization": "L1 (lasso)",
        "C": [0.01, 0.1, 1, 10],
        "random_state": SEED
    },

    "classification_tree": {
        "max_depth": [3, 5, 8],
        "min_samples_leaf": [25, 50, 100],
        "random_state": SEED
    },

    "random_forest": {
        "random_state": SEED
        # Keep all other model settings at their defaults.
    }
}

In [20]:
#PART C: REUSES Y AND CV_SPLITS FROM EARLIER, ENSURES IDENTICAL EVALUATION SPLITS

import numpy as np
import pandas as pd

from sklearn.metrics import (
    log_loss,
    accuracy_score,
    precision_score,
    recall_score
)

results = []

for fold, (train_index, validation_index) in enumerate(
    cv_splits, start=1
):
    y_train = y.iloc[train_index]
    y_validation = y.iloc[validation_index]

    # Mean of a 0/1 variable = proportion of ones.
    # Use only the training portion to calculate this.
    baseline_probability = y_train.mean()

    # Give every validation title the same probability.
    probabilities = np.full(
        len(y_validation),
        baseline_probability
    )

    # Use a 0.5 threshold for the secondary classification metrics.
    predictions = (probabilities >= 0.5).astype(int)

    # Precision is undefined if there are no positive predictions.
    precision = (
        precision_score(y_validation, predictions)
        if predictions.sum() > 0
        else np.nan
    )

    results.append({
        "Fold": fold,
        "Training Netflix %": 100 * baseline_probability,
        "Log loss": log_loss(
            y_validation, probabilities, labels=[0, 1]
        ),
        "Misclassification %": 100 * (
            1 - accuracy_score(y_validation, predictions)
        ),
        "Precision %": 100 * precision,
        "Recall %": 100 * recall_score(
            y_validation, predictions, zero_division=0
        )
    })

baseline_results = pd.DataFrame(results)

print("RESULTS FOR EACH FOLD")
print(baseline_results.round(6).to_string(index=False))

print("\nAVERAGE PERFORMANCE ACROSS FOLDS")
metrics = [
    "Log loss",
    "Misclassification %",
    "Precision %",
    "Recall %"
]
print(baseline_results[metrics].mean().round(6))

print("\nNaN precision means undefined, not zero.")

RESULTS FOR EACH FOLD
 Fold  Training Netflix %  Log loss  Misclassification %  Precision %  Recall %
    1           37.715246  0.662784            37.741394          NaN       0.0
    2           37.721785  0.662653            37.715246          NaN       0.0
    3           37.721785  0.662653            37.715246          NaN       0.0
    4           37.721785  0.662653            37.715246          NaN       0.0
    5           37.721785  0.662653            37.715246          NaN       0.0

AVERAGE PERFORMANCE ACROSS FOLDS
Log loss                0.662679
Misclassification %    37.720475
Precision %                  NaN
Recall %                0.000000
dtype: float64

NaN precision means undefined, not zero.


In [21]:
#PART D: DEFINE TWO FEATURE SETS
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

genre_columns = [c for c in X.columns if c.startswith("genre_")]

shared_binary = [
    "Type", "runtime_missing", "genres_missing"
] + genre_columns

feature_sets = {
    "Set 1": {
        "numeric": ["runtime_minutes"],
        "binary": shared_binary,
        "categorical": ["Age"],
    },
    "Set 2": {
        "numeric": [
            "runtime_minutes",
            "imdb_rating",
            "rt_score",
            "imdb_num_votes",
        ],
        "binary": shared_binary,
        "categorical": ["Age"],
    },
}

# Keep all genre indicators; no frequency cutoff.
# Year remains excluded pending clarification from the TA.
print(f"Both sets include all {len(genre_columns)} genre indicators.")

Both sets include all 28 genre indicators.


In [22]:
#PART D: PREPARE VARIABLES (converts Rotten Tomatoes scores into numbers and creates missing-genre indicator)

class PrepareTitleFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, genre_columns):
        self.genre_columns = genre_columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        out = X.copy().replace(r"^\s*$", np.nan, regex=True)

        out["Age"] = out["Age"].fillna("unrated")

        out["genres_missing"] = (
            out["genres"].fillna("").str.strip().eq("").astype(int)
        )

        out.loc[
            out["genres_missing"].eq(1), self.genre_columns
        ] = 0

        # Convert "66/100" to 66, for example.
        out["rt_score"] = pd.to_numeric(
            out["Rotten Tomatoes"].astype("string")
                .str.replace("/100", "", regex=False),
            errors="coerce",
        )

        for column in ["imdb_rating", "imdb_num_votes"]:
            out[column] = pd.to_numeric(out[column], errors="coerce")

        return out

In [23]:
#PART D: DEFINE THE THREE ALGORITHMS

models = {
    "Logistic regression": LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=2000,
        random_state=SEED,
    ),

    "Classification tree": DecisionTreeClassifier(
        max_depth=5,           # Starting value; tune in part (e).
        min_samples_leaf=50,  # Starting value; tune in part (e).
        random_state=SEED,
    ),

    "Random forest": RandomForestClassifier(
        random_state=SEED,    # All other settings remain at defaults.
    ),
}

param_grids = {
    "Logistic regression": {
        "model__C": [0.01, 0.1, 1, 10],
    },

    "Classification tree": {
        "model__max_depth": [3, 5, 8],
        # These are minimum leaf sizes, not genre-frequency cutoffs.
        "model__min_samples_leaf": [25, 50, 100],
    },

    "Random forest": {},  # No tuning, following office-hours advice.
}

In [24]:
#PART D: COMBINE PREPROCESSING WITH EACH MODEL (a pipeline bundles preparation & modeling together)

def build_model_pipeline(set_name, model_name):
    features = feature_sets[set_name]

    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()
         if model_name == "Logistic regression" else "passthrough"),
    ])

    encoding = ColumnTransformer([
        ("numeric", numeric_steps, features["numeric"]),

        ("age", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ), features["categorical"]),

        ("binary", "passthrough", features["binary"]),
    ], remainder="drop")

    return Pipeline([
        ("prepare", PrepareTitleFeatures(genre_columns)),
        ("runtime", RuntimeMedianImputer()),  # From part (b).
        ("encoding", encoding),
        ("model", clone(models[model_name])),
    ])


pipelines = {
    (set_name, model_name): build_model_pipeline(set_name, model_name)
    for set_name in feature_sets
    for model_name in models
}

for set_name, model_name in pipelines:
    print(f"{set_name}: {model_name}")

Set 1: Logistic regression
Set 1: Classification tree
Set 1: Random forest
Set 2: Logistic regression
Set 2: Classification tree
Set 2: Random forest


In [25]:
#DEFINE EVALUATION METRICS (log loss selects the settings; other metrics use fixed threshold 0.5)

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (
    log_loss, accuracy_score, precision_score, recall_score
)
from IPython.display import display

THRESHOLD = 0.5

def score_predictions(y_true, probabilities):
    predictions = (probabilities >= THRESHOLD).astype(int)

    return {
        "Log loss": log_loss(y_true, probabilities, labels=[0, 1]),
        "Error (%)": 100 * (1 - accuracy_score(y_true, predictions)),
        "Precision (%)": (
            100 * precision_score(y_true, predictions)
            if predictions.sum() > 0 else np.nan
        ),
        "Recall (%)": 100 * recall_score(
            y_true, predictions, zero_division=0
        ),
    }

def summarize_folds(records):
    scores = pd.DataFrame(records)

    return {
        "Log loss": scores["Log loss"].mean(),
        "Log loss SD": scores["Log loss"].std(ddof=1),
        "Error (%)": scores["Error (%)"].mean(),

        # Preserve undefined precision if any fold has no positive predictions.
        "Precision (%)": (
            scores["Precision (%)"].mean()
            if scores["Precision (%)"].notna().all() else np.nan
        ),

        "Recall (%)": scores["Recall (%)"].mean(),
    }

def settings_text(parameters):
    return "; ".join(
        f"{key.replace('model__', '')}={value}"
        for key, value in parameters.items()
    ) or "Defaults; random_state=226"

In [26]:
#PART D: train and evaluate every combination (main training block; everything starts with a fresh pipeline so imputation, encoding, and scaling are learned within that fold's training portion)

candidate_rows = []
fold_rows = []

total_candidates = sum(
    len(ParameterGrid(param_grids[model_name]))
    for set_name, model_name in pipelines
)

print(
    f"{total_candidates} settings across all combinations; "
    f"{total_candidates * len(cv_splits)} model fits."
)

for (set_name, model_name), template in pipelines.items():
    grid = list(ParameterGrid(param_grids[model_name]))

    print(
        f"\n{set_name} | {model_name}: {len(grid)} settings",
        flush=True,
    )

    for candidate_id, parameters in enumerate(grid, start=1):
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
            estimator = clone(template).set_params(**parameters)

            # Fit preprocessing and model using training rows only.
            estimator.fit(X.iloc[train_idx], y.iloc[train_idx])

            # Probability that Netflix = 1.
            positive_column = np.flatnonzero(estimator.classes_ == 1)[0]
            probabilities = estimator.predict_proba(
                X.iloc[valid_idx]
            )[:, positive_column]

            metrics = score_predictions(
                y.iloc[valid_idx], probabilities
            )

            fold_scores.append(metrics)

            fold_rows.append({
                "Feature set": set_name,
                "Algorithm": model_name,
                "Candidate": candidate_id,
                "Fold": fold,
                **metrics,
            })

        candidate_rows.append({
            "Feature set": set_name,
            "Algorithm": model_name,
            "Candidate": candidate_id,
            "Parameters": parameters,
            **summarize_folds(fold_scores),
        })

        print(
            f"  Finished setting {candidate_id}/{len(grid)}",
            flush=True,
        )

# Retain all candidate results and individual fold results.
search_results = pd.DataFrame(candidate_rows)
fold_results = pd.DataFrame(fold_rows)

# Select settings separately for each algorithm/feature-set combination.
best_indices = search_results.groupby(
    ["Feature set", "Algorithm"], sort=False
)["Log loss"].idxmin()

best_results = search_results.loc[best_indices].copy()

28 settings across all combinations; 140 model fits.

Set 1 | Logistic regression: 4 settings
  Finished setting 1/4
  Finished setting 2/4
  Finished setting 3/4
  Finished setting 4/4

Set 1 | Classification tree: 9 settings
  Finished setting 1/9
  Finished setting 2/9
  Finished setting 3/9
  Finished setting 4/9
  Finished setting 5/9
  Finished setting 6/9
  Finished setting 7/9
  Finished setting 8/9
  Finished setting 9/9

Set 1 | Random forest: 1 settings
  Finished setting 1/1

Set 2 | Logistic regression: 4 settings
  Finished setting 1/4
  Finished setting 2/4
  Finished setting 3/4
  Finished setting 4/4

Set 2 | Classification tree: 9 settings
  Finished setting 1/9
  Finished setting 2/9
  Finished setting 3/9
  Finished setting 4/9
  Finished setting 5/9
  Finished setting 6/9
  Finished setting 7/9
  Finished setting 8/9
  Finished setting 9/9

Set 2 | Random forest: 1 settings
  Finished setting 1/1


In [27]:
#PART D: PRODUCE COMPARISON TABLE

baseline_fold_scores = []

for train_idx, valid_idx in cv_splits:
    training_proportion = y.iloc[train_idx].mean()
    probabilities = np.full(len(valid_idx), training_proportion)

    baseline_fold_scores.append(
        score_predictions(y.iloc[valid_idx], probabilities)
    )

baseline_summary = summarize_folds(baseline_fold_scores)

comparison = best_results.drop(
    columns=["Candidate", "Parameters"]
).copy()

comparison["Settings tried"] = comparison["Algorithm"].map(
    lambda name: settings_text(param_grids[name])
)

comparison["Settings selected"] = best_results["Parameters"].map(
    settings_text
)

comparison["Candidates tried"] = comparison["Algorithm"].map(
    lambda name: len(ParameterGrid(param_grids[name]))
)

baseline_row = {
    "Feature set": "None",
    "Algorithm": "Baseline",
    **baseline_summary,
    "Settings tried": "Training Netflix proportion",
    "Settings selected": "Training Netflix proportion",
    "Candidates tried": 1,
}

comparison = pd.concat(
    [pd.DataFrame([baseline_row]), comparison],
    ignore_index=True,
)

# Positive = lower (better) log loss than the baseline.
comparison["Improvement vs baseline"] = (
    baseline_summary["Log loss"] - comparison["Log loss"]
)

column_order = [
    "Feature set",
    "Algorithm",
    "Log loss",
    "Log loss SD",
    "Improvement vs baseline",
    "Error (%)",
    "Precision (%)",
    "Recall (%)",
    "Candidates tried",
    "Settings tried",
    "Settings selected",
]

comparison = comparison[column_order]

with pd.option_context("display.max_colwidth", 120):
    display(comparison.round(6))

,Feature set,Algorithm,Log loss,Log loss SD,Improvement vs baseline,Error (%),Precision (%),Recall (%),Candidates tried,Settings tried,Settings selected
0,None,Baseline,0.662679,0.000059,0.000000,37.720475,NaN,0.000000,1,Training Netflix proportion,Training Netflix proportion
1,Set 1,Logistic regression,0.644774,0.003700,0.017905,38.006010,48.087793,7.770847,4,"C=[0.01, 0.1, 1, 10]",C=1
2,Set 1,Classification tree,0.625808,0.008839,0.036871,35.083108,55.395873,37.075097,9,"max_depth=[3, 5, 8]; min_samples_leaf=[25, 50, 100]",max_depth=8; min_samples_leaf=50
3,Set 1,Random forest,0.845658,0.051676,-0.182979,37.266930,50.763604,45.446661,1,Defaults; random_state=226,Defaults; random_state=226
4,Set 2,Logistic regression,0.632040,0.003914,0.030639,37.056839,52.283919,20.952927,4,"C=[0.01, 0.1, 1, 10]",C=1
5,Set 2,Classification tree,0.619075,0.005666,0.043604,35.486363,55.822087,31.929043,9,"max_depth=[3, 5, 8]; min_samples_leaf=[25, 50, 100]",max_depth=5; min_samples_leaf=25
6,Set 2,Random forest,0.689221,0.031949,-0.026542,32.613846,58.473030,46.670862,1,Defaults; random_state=226,Defaults; random_state=226


In [28]:
#PART D: PRINT COMPARISONS FOR REPORT & INTERPRETATION

print("Lowest mean validation log loss within each feature set:")

for set_name, group in best_results.groupby("Feature set", sort=False):
    best = group.loc[group["Log loss"].idxmin()]

    print(
        f"{set_name}: {best['Algorithm']} "
        f"(log loss = {best['Log loss']:.6f})"
    )

loss_by_set = best_results.pivot(
    index="Algorithm",
    columns="Feature set",
    values="Log loss",
)

print("\nEffect of adding ratings/popularity:")

for algorithm, row in loss_by_set.iterrows():
    reduction = row["Set 1"] - row["Set 2"]

    print(
        f"{algorithm}: {row['Set 1']:.6f} → {row['Set 2']:.6f}; "
        f"reduction in log loss = {reduction:+.6f}"
    )

print("\nPositive reduction = Set 2 performed better.")

Lowest mean validation log loss within each feature set:
Set 1: Classification tree (log loss = 0.625808)
Set 2: Classification tree (log loss = 0.619075)

Effect of adding ratings/popularity:
Classification tree: 0.625808 → 0.619075; reduction in log loss = +0.006733
Logistic regression: 0.644774 → 0.632040; reduction in log loss = +0.012734
Random forest: 0.845658 → 0.689221; reduction in log loss = +0.156437

Positive reduction = Set 2 performed better.
